# Phase 04. Identify, then intervene.

*Where does the prompt→target signal actually live, and what is the smallest intervention that can exploit it?*

## Research statement

Notebook 03 demonstrated that the auxiliary JEPA loss can fall without improving generation. The corrected family-diverse audit changed the diagnosis: target representations are not razor-thin, and the raw/predicted aligned-vs-shuffled gaps are small but nonzero. The failure is narrower and more useful than "no signal exists": the predictor learns a dominant common direction, remains directionally collapsed, and recovers only a weak pair-specific component. Generation still stays at 0/100 exact match.

This notebook does not begin by training. It first identifies whether that weak cross-view signal is real, where it lives across `(layer, pooling)` combinations, and whether it survives increasingly hostile falsification tests.

The hypothesis is not "JEPA improves Qwen." It is:

> **A compact component of the prompt representation contains enough information to predict the pair-specific component of the SQL representation, and encouraging this mapping can improve task performance without materially degrading autoregressive learning.**

That is narrow enough to falsify.

## The central decomposition

The failure in nb03 suggests target embeddings decompose as:

$$
z^{T}_i = \underbrace{\mu}_{\text{global terminal direction}} + \underbrace{\alpha_{g(i)}}_{\text{SQL template family}} + \underbrace{s_i}_{\text{pair-specific semantics}} + \varepsilon_i
$$

The nb03 predictor mostly learned $\mu$ and only weakly recovered $s_i$. The earlier one-family audit underestimated the target manifold; the corrected family-diverse audit shows real between-row variance, but the learned predictor still collapses into a narrow directional cone. What JEPA needs is a recoverable relationship between the prompt and $s_i$ that is not swamped by $\mu$ or $\alpha_g$.

This notebook asks four ordered questions:

$$
\begin{aligned}
Q_1 &: \operatorname{Var}(z^T_i) > 0\text{?} \\
Q_2 &: z^T_i \text{ contains pair-specific rather than only family information?} \\
Q_3 &: z^T_i \text{ is predictable from } z^P_i \text{ out of sample?} \\
Q_4 &: \text{forcing that prediction improves generation without crowding out CE?}
\end{aligned}
$$

Training is prohibited until $Q_1$, $Q_2$, $Q_3$ pass. If they don't, the notebook stops and concludes that no usable prompt→target representation was identified on this data. That is a valid result.

## Scope of this notebook

This notebook covers everything through the representation selection gate: contract → data → latent panel → geometry, retrieval, frozen predictability, FWL. That's up through $Q_3$. If the gate produces a passing candidate `(layer, pooling)` (with optional Residual JEPA $K$), a subsequent notebook adds the intervention lane: gradient-calibrated raw + Residual JEPA arms, screening across three seeds, confirmation across five, decision engine.


In [ ]:
# Imports and deterministic environment

import hashlib
import itertools
import json
import math
import os
import random
import subprocess
import sys
import time
from collections import Counter, defaultdict
from dataclasses import dataclass, field, asdict
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

# Path resolution: prefer the notebook experiment directory, not the project
# root. The project root also contains a small legacy data/nl_sql.jsonl, but
# the Phase 04 experiment needs notebooks/data plus the local Qwen model.
HERE = Path.cwd().resolve()

def _resolve_notebook_dir(start):
    candidates = []
    cur = start
    while True:
        candidates.append(cur)
        candidates.append(cur / "notebooks")
        if cur == cur.parent:
            break
        cur = cur.parent

    seen = set()
    for cand in candidates:
        cand = cand.resolve()
        if cand in seen:
            continue
        seen.add(cand)
        data = cand / "data" / "nl_sql.jsonl"
        model_config = cand / "model" / "Qwen2.5-0.5B-Instruct" / "config.json"
        if data.exists() and model_config.exists():
            return cand

    raise FileNotFoundError(
        "Could not locate notebook experiment directory. Expected both "
        "data/nl_sql.jsonl and model/Qwen2.5-0.5B-Instruct/config.json. "
        f"Started from {start}."
    )

NOTEBOOK_DIR = _resolve_notebook_dir(HERE)
os.chdir(NOTEBOOK_DIR)

DATA_PATH   = NOTEBOOK_DIR / "data" / "nl_sql.jsonl"
MODEL_DIR   = NOTEBOOK_DIR / "model" / "Qwen2.5-0.5B-Instruct"
OUTPUT_DIR  = NOTEBOOK_DIR / "04-outputs"
CACHE_DIR   = OUTPUT_DIR / "latent-cache"
MANIFEST_DIR = OUTPUT_DIR / "manifests"
GEOMETRY_DIR = OUTPUT_DIR / "geometry"

assert DATA_PATH.exists(), f"missing data file: {DATA_PATH}"
assert (MODEL_DIR / "config.json").exists(), f"missing local model config: {MODEL_DIR / 'config.json'}"
assert DATA_PATH.stat().st_size > 1_000_000, (
    f"{DATA_PATH} is too small for the Phase 04 14k-row experiment. "
    "This usually means the notebook resolved to the project-root toy dataset."
)

for d in (OUTPUT_DIR, CACHE_DIR, MANIFEST_DIR, GEOMETRY_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Environment stamps
def _git_commit(repo):
    try:
        return subprocess.check_output(
            ["git", "-C", str(repo), "rev-parse", "HEAD"], text=True
        ).strip()
    except Exception:
        return None

def _git_dirty(repo):
    try:
        out = subprocess.check_output(
            ["git", "-C", str(repo), "status", "--porcelain"], text=True
        )
        return bool(out.strip())
    except Exception:
        return None

_import_versions = {
    "python":       sys.version.split()[0],
    "torch":        torch.__version__,
    "numpy":        np.__version__,
}
try:
    import transformers
    _import_versions["transformers"] = transformers.__version__
except Exception:
    pass

ENV = {
    "python":            _import_versions["python"],
    "torch":             _import_versions["torch"],
    "numpy":             _import_versions["numpy"],
    "transformers":      _import_versions.get("transformers"),
    "cuda_available":    torch.cuda.is_available(),
    "cuda_device":       torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "cuda_capability":   torch.cuda.get_device_capability(0) if torch.cuda.is_available() else None,
    "cuda_total_gb":     round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2) if torch.cuda.is_available() else None,
    "git_commit":        _git_commit(NOTEBOOK_DIR),
    "git_dirty":         _git_dirty(NOTEBOOK_DIR),
}

print(json.dumps(ENV, indent=2, default=str))
print()
print(f"NOTEBOOK_DIR : {NOTEBOOK_DIR}")
print(f"DATA_PATH    : {DATA_PATH}  exists: {DATA_PATH.exists()}  size={DATA_PATH.stat().st_size:,}")
print(f"MODEL_DIR    : {MODEL_DIR}  exists: {MODEL_DIR.exists()}")
print(f"OUTPUT_DIR   : {OUTPUT_DIR}")


## Preregistered dashboard

Everything downstream reacts to what's set here. `λ` is deliberately not fixed — it will be calibrated to a target gradient ratio in the training notebook rather than chosen because "0.1 looks normal."

In [ ]:
# Preregistered dashboard

# ─── Reproducibility ───────────────────────────────────────────────────────
MASTER_SEED     = 1337
SCREEN_SEEDS    = [1337, 2027, 8675]                    # used in the training notebook
CONFIRM_SEEDS   = [1337, 2027, 8675, 4211, 9917]        # used in the training notebook

# These versions are part of CONFIG_HASH. Bump one whenever rendered views,
# content masks, or cache schema change, so stale latent panels cannot be
# silently reused after a logic edit.
VIEW_CONTRACT_VERSION = "qwen-chat-target-includes-prompt-v3"
CONTENT_MASK_VERSION  = "qwen-role-span-eot-newline-excluded-v2"
EXTRACTOR_VERSION     = "latent-panel-iid-ood-ridge-gates-v3"

# ─── Model ─────────────────────────────────────────────────────────────────
MODEL_NAME      = str(MODEL_DIR)
DEVICE          = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE           = torch.bfloat16 if (DEVICE == "cuda" and torch.cuda.is_bf16_supported()) else torch.float32
SEQ_LEN         = 1024

# ─── Data splits ───────────────────────────────────────────────────────────
# The counts below define the latent panel used for representation
# identification. They are separate from any nb03/nb05 training split.
N_LATENT_TRAIN         = 3000
N_LATENT_CALIBRATION   = 750
N_LATENT_TEST          = 750
IID_HOLDOUT_FRACTION   = 0.15   # of within-family rows, per family
OOD_TEMPLATE_FRACTION  = 0.20   # fraction of template families reserved for OOD test

# ─── Candidate representations ─────────────────────────────────────────────
LAYERS    = [-1, -4, -8]
POOLINGS  = ["terminal", "last_content", "content_mean", "content_tail4"]

# ─── Frozen predictability ─────────────────────────────────────────────────
RIDGE_ALPHAS     = np.logspace(-4, 4, 9)
N_PERMUTATIONS   = 250
TRAIN_VAL_SPLIT  = 0.8            # within N_LATENT_TRAIN, fraction used for ridge fit
                                  # (rest = validation for ridge alpha selection)

# ─── Selection gate thresholds ─────────────────────────────────────────────
# All applied to CALIBRATION set only; the test set stays locked until the training notebook.
GATE_ALIGNED_ABOVE_NULL_PCTILE     = 95     # aligned R² must beat this pctile of derangement null
GATE_MIN_RETRIEVAL_MRR             = 0.02   # predicted-target MRR must beat this
GATE_MIN_CONSTANT_PRED_ADVANTAGE   = 0.05   # predicted-target cos - constant_baseline_cos ≥ this
GATE_MIN_EFFECTIVE_RANK_RATIO      = 1.5    # effective_rank vs terminal-representation baseline
GATE_ONE_SE_SELECTION              = True   # among passing candidates, prefer cheapest within 1 SE

# ─── Intervention (deferred to the training notebook) ─────────────────────
ARM                       = "none"           # none | raw_jepa | residual_jepa | shuffled
COMMON_COMPONENTS_TO_REMOVE = [0, 1, 4, 8]   # K for Residual JEPA; selected on calibration only
PILOT_STEPS               = 150
FULL_STEPS                = 750
TARGET_AUX_GRAD_RATIO     = 0.10

BATCH_SIZE                = 1
GRAD_ACCUM_STEPS          = 16
LEARNING_RATE             = 2.5e-6
WEIGHT_DECAY              = 0.08

print("Dashboard loaded.")
print(f"  Latent panel : {N_LATENT_TRAIN} train + {N_LATENT_CALIBRATION} calib + {N_LATENT_TEST} test")
print(f"  Sweep        : {len(LAYERS)} layers × {len(POOLINGS)} poolings = {len(LAYERS)*len(POOLINGS)} combos")
print(f"  Device/dtype : {DEVICE} / {DTYPE}")
print(f"  Master seed  : {MASTER_SEED}")
print(f"  View/mask/extractor versions: {VIEW_CONTRACT_VERSION} | {CONTENT_MASK_VERSION} | {EXTRACTOR_VERSION}")


## Immutable run manifest

Before any result appears, freeze the experimental contract: which hypotheses, which candidate representations, which split rule, which primary outcomes. Every arm downstream (in later notebooks) must reference this manifest hash. If any downstream code changes the split, the config hash changes and old results are marked stale.

In [ ]:
# Immutable run manifest

MANIFEST = {
    "notebook":   "04-identify-then-intervene.ipynb",
    "scope":      "contract through selection gate (training deferred)",
    "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    "hypothesis": (
        "A compact component of the prompt representation contains enough information "
        "to predict the pair-specific component of the SQL representation, and "
        "encouraging this mapping can improve task performance without materially "
        "degrading autoregressive learning."
    ),
    "sub_questions": [
        "Q1: Var(z_T) > 0?",
        "Q2: z_T contains pair-specific rather than only family information?",
        "Q3: z_T is predictable from z_P out of sample?",
        "Q4: forcing that prediction improves generation without crowding out CE? [deferred to next notebook]",
    ],
    "candidate_representations": [
        {"layer": ell, "pooling": p} for ell in LAYERS for p in POOLINGS
    ],
    "selection_rule": (
        f"Raw pass: ridge-predicted target vectors must (1) beat the derangement-null "
        f"aligned-R² at pctile ≥ {GATE_ALIGNED_ABOVE_NULL_PCTILE}, "
        f"(2) exceed predicted retrieval MRR = {GATE_MIN_RETRIEVAL_MRR}, "
        f"(3) exceed constant-predictor cos advantage = {GATE_MIN_CONSTANT_PRED_ADVANTAGE}, "
        f"(4) exceed effective-rank ratio = {GATE_MIN_EFFECTIVE_RANK_RATIO}. "
        f"FWL pass: residualized R² must beat its own derangement null at the same percentile. "
        f"Among passers, apply the 1-SE rule and pick the cheapest representation."
    ),
    "primary_outcomes": [
        "ridge_predicted_R2_out_of_sample",
        "ridge_predicted_R2_iid_calibration",
        "ridge_predicted_R2_ood_calibration",
        "ridge_predicted_MRR",
        "ridge_predicted_constant_cos_advantage",
        "effective_rank",
        "residualized_R2_within_family",
        "raw_gate_passed",
        "fwl_gate_passed",
        "gate_branch",
    ],
    "seed":  MASTER_SEED,
    "env":   ENV,
    "config": {
        "MODEL_NAME": MODEL_NAME,
        "SEQ_LEN": SEQ_LEN,
        "VIEW_CONTRACT_VERSION": VIEW_CONTRACT_VERSION,
        "CONTENT_MASK_VERSION": CONTENT_MASK_VERSION,
        "EXTRACTOR_VERSION": EXTRACTOR_VERSION,
        "N_LATENT_TRAIN": N_LATENT_TRAIN,
        "N_LATENT_CALIBRATION": N_LATENT_CALIBRATION,
        "N_LATENT_TEST": N_LATENT_TEST,
        "IID_HOLDOUT_FRACTION": IID_HOLDOUT_FRACTION,
        "OOD_TEMPLATE_FRACTION": OOD_TEMPLATE_FRACTION,
        "LAYERS": LAYERS,
        "POOLINGS": POOLINGS,
        "RIDGE_ALPHAS": [float(a) for a in RIDGE_ALPHAS],
        "N_PERMUTATIONS": N_PERMUTATIONS,
        "TRAIN_VAL_SPLIT": TRAIN_VAL_SPLIT,
        "GATE_ALIGNED_ABOVE_NULL_PCTILE": GATE_ALIGNED_ABOVE_NULL_PCTILE,
        "GATE_MIN_RETRIEVAL_MRR": GATE_MIN_RETRIEVAL_MRR,
        "GATE_MIN_CONSTANT_PRED_ADVANTAGE": GATE_MIN_CONSTANT_PRED_ADVANTAGE,
        "GATE_MIN_EFFECTIVE_RANK_RATIO": GATE_MIN_EFFECTIVE_RANK_RATIO,
    },
}

_config_bytes = json.dumps(MANIFEST["config"], sort_keys=True).encode()
CONFIG_HASH = hashlib.sha256(_config_bytes).hexdigest()[:12]
MANIFEST["config_hash"] = CONFIG_HASH

_manifest_path = MANIFEST_DIR / f"experiment_manifest_{CONFIG_HASH}.json"
_manifest_path.write_text(json.dumps(MANIFEST, indent=2, default=str))
print(f"Manifest saved → {_manifest_path}")
print(f"CONFIG_HASH    = {CONFIG_HASH}")
print()
print("Downstream arms MUST reference this hash. If dashboard values change,")
print("the hash changes and prior results are marked stale.")


## Reconstruct the data-generating process

14,000 rows is not 14,000 independent semantic observations. Before any representation work, quantify the SQL template concentration and set up split regimes that respect it.

In [ ]:
# Load pairs and derive template families

def load_pairs(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            rows.append({"row_id": i, "prompt": obj["prompt"], "target": obj["target"]})
    return rows

def sql_family_signature(sql):
    """Structural signature: strip literals, dates, numbers, state codes.

    Two rows share a signature iff they are the same SQL query up to substituted
    literals/dates/numbers/state codes. This is an econometric family key -- rows
    with the same signature are near-replicates of each other.

    NB: text-based normalization, not AST. Text-based works well for the templated
    generator that produced this dataset; upgrade to sqlparse/sqlglot if the
    family concentration below looks suspiciously high or low.
    """
    s = sql.strip().lower()
    s = re.sub(r"'\d{4}-\d{2}-\d{2}'", "'<date>'", s)
    s = re.sub(r"'\d{4}-\d{2}'", "'<yearmonth>'", s)
    s = re.sub(r"date\s+'[^']*'", "date '<date>'", s)
    s = re.sub(r"'[a-z]{2}'", "'<state>'", s)
    s = re.sub(r"'[^']*'", "'<literal>'", s)
    s = re.sub(r"\b\d{4}-\d{2}-\d{2}\b", "<date>", s)
    s = re.sub(r"\b\d+\.\d+\b", "<num>", s)
    s = re.sub(r"\b\d+\b", "<int>", s)
    s = re.sub(r"\s+", " ", s)
    return s

import re
pairs = load_pairs(DATA_PATH)
print(f"Loaded {len(pairs):,} pairs from {DATA_PATH}")

for p in pairs:
    p["template_id"] = sql_family_signature(p["target"])
    p["prompt_length"] = len(p["prompt"])
    p["target_length"] = len(p["target"])

# Assign a scenario_id: coarser than template_id, groups rows sharing the same
# tables (regardless of columns/predicates). Simple heuristic: sorted FROM tables.
def _scenario(sql):
    s = sql.lower()
    from_ = re.search(r"\bfrom\b(.+?)(?:\bwhere\b|\border by\b|\bgroup by\b|;|$)", s, re.DOTALL)
    if not from_:
        return "<unknown>"
    body = from_.group(1)
    tables = re.findall(r"\b([a-z_][a-z0-9_]*)\b(?:\s+[a-z][a-z0-9_]*)?\s*(?:join|,|on|$|\bwhere\b|\border|\bgroup)", body + " ")
    # Fallback: just extract 'from X ...' first table
    if not tables:
        m = re.search(r"from\s+([a-z_][a-z0-9_]*)", body)
        tables = [m.group(1)] if m else []
    seen = []
    for t in tables:
        if t in {"on", "and", "or", "where", "join", "left", "right", "inner", "outer"}:
            continue
        if t not in seen:
            seen.append(t)
    return "|".join(sorted(seen)) if seen else "<unknown>"

for p in pairs:
    p["scenario_id"] = _scenario(p["target"])

sample = pairs[0]
print()
print(f"Sample row_id={sample['row_id']}")
print(f"  prompt      : {sample['prompt'][:120]}...")
print(f"  target[:120]: {sample['target'][:120]}...")
print(f"  template_id : {sample['template_id'][:120]}...")
print(f"  scenario_id : {sample['scenario_id']}")

In [ ]:
# Family concentration audit

family_sizes = Counter(p["template_id"] for p in pairs)
scenario_sizes = Counter(p["scenario_id"] for p in pairs)

n = len(pairs)
n_exact_targets = len(set(p["target"] for p in pairs))
n_families = len(family_sizes)
n_scenarios = len(scenario_sizes)

sizes = np.array(sorted(family_sizes.values(), reverse=True), dtype=float)
shares = sizes / sizes.sum()
hhi = float((shares ** 2).sum())
inv_hhi_eff_families = 1.0 / hhi
top4_share = shares[:4].sum() if len(shares) >= 4 else shares.sum()
top8_share = shares[:8].sum() if len(shares) >= 8 else shares.sum()

print(f"Rows                                     : {n:,}")
print(f"Distinct exact targets                   : {n_exact_targets:,}")
print(f"Distinct template families (signatures)  : {n_families:,}")
print(f"Distinct scenarios (FROM-table sets)     : {n_scenarios:,}")
print(f"Herfindahl (families)                    : {hhi:.4f}")
print(f"Effective # of balanced families (1/HHI) : {inv_hhi_eff_families:.1f}")
print(f"Top-4 family cumulative share            : {top4_share:.1%}")
print(f"Top-8 family cumulative share            : {top8_share:.1%}")
print()
print("Top 15 families by row count:")
for i, (fam, count) in enumerate(family_sizes.most_common(15), 1):
    print(f"  {i:2}. n={count:5}  {fam[:110]}...")

# Save distribution
(GEOMETRY_DIR / "family_distribution.json").write_text(json.dumps({
    "n_rows": n,
    "n_exact_targets": n_exact_targets,
    "n_families": n_families,
    "n_scenarios": n_scenarios,
    "hhi": hhi,
    "inv_hhi_eff_families": inv_hhi_eff_families,
    "top_family_sizes": [{"template": t, "n": c} for t, c in family_sizes.most_common(50)],
}, indent=2))
print()
print(f"Saved → {GEOMETRY_DIR / 'family_distribution.json'}")

In [ ]:
# Create two evaluation regimes

# Both estimands share a common training set; test sets differ.
#
#   IID interpolation : known SQL template, unseen literal combination
#                       -> holdout rows sampled uniformly WITHIN each family
#   Template OOD      : entire SQL skeleton held out
#                       -> holdout rows come from ENTIRE families reserved for test
#
# The latent panel (train/calibration/test) is drawn from these regimes:
#   latent train   = subset of the joint train pool
#   latent calib   = rows from small-N iid-holdout families + small-N ood-holdout families
#   latent test    = disjoint rows from same
#
# Split RNG is master-seeded; the split manifest lists exact row_ids.

_rng = random.Random(MASTER_SEED)

# Group rows by family
by_family = defaultdict(list)
for p in pairs:
    by_family[p["template_id"]].append(p["row_id"])

families = list(by_family.keys())
_rng.shuffle(families)

# Reserve OOD families
n_ood_families = max(1, int(len(families) * OOD_TEMPLATE_FRACTION))
ood_families = set(families[:n_ood_families])
iid_families = [f for f in families if f not in ood_families]

# Within iid_families: reserve IID_HOLDOUT_FRACTION per family
train_row_ids = set()
iid_holdout_row_ids = set()

for fam in iid_families:
    rows = by_family[fam].copy()
    _rng.shuffle(rows)
    n_hold = max(0, int(len(rows) * IID_HOLDOUT_FRACTION))
    iid_holdout_row_ids.update(rows[:n_hold])
    train_row_ids.update(rows[n_hold:])

ood_row_ids = set()
for fam in ood_families:
    ood_row_ids.update(by_family[fam])

# Sanity: partition
assert train_row_ids.isdisjoint(iid_holdout_row_ids), "train ∩ iid_holdout not empty"
assert train_row_ids.isdisjoint(ood_row_ids), "train ∩ ood not empty"
assert iid_holdout_row_ids.isdisjoint(ood_row_ids), "iid_holdout ∩ ood not empty"

print(f"Total pairs             : {len(pairs):,}")
print(f"OOD families reserved   : {len(ood_families)} / {len(families)}")
print(f"Train row_ids           : {len(train_row_ids):,}")
print(f"IID-holdout row_ids     : {len(iid_holdout_row_ids):,}")
print(f"OOD-holdout row_ids     : {len(ood_row_ids):,}")

# Sample latent panel
_rng2 = random.Random(MASTER_SEED + 1)

train_list = sorted(train_row_ids)
iid_hold_list = sorted(iid_holdout_row_ids)
ood_hold_list = sorted(ood_row_ids)

_rng2.shuffle(train_list)
_rng2.shuffle(iid_hold_list)
_rng2.shuffle(ood_hold_list)

# latent_train : from train_list
n_lt = min(N_LATENT_TRAIN, len(train_list))
latent_train_ids = train_list[:n_lt]

# latent_calibration : half from iid_hold, half from ood_hold
n_lc_iid = min(N_LATENT_CALIBRATION // 2, len(iid_hold_list))
n_lc_ood = min(N_LATENT_CALIBRATION - n_lc_iid, len(ood_hold_list))
latent_calibration_ids = iid_hold_list[:n_lc_iid] + ood_hold_list[:n_lc_ood]
latent_calibration_regimes = ["iid"] * n_lc_iid + ["ood"] * n_lc_ood

# latent_test : disjoint remainder
n_lt_iid = min(N_LATENT_TEST // 2, len(iid_hold_list) - n_lc_iid)
n_lt_ood = min(N_LATENT_TEST - n_lt_iid, len(ood_hold_list) - n_lc_ood)
latent_test_ids = (
    iid_hold_list[n_lc_iid : n_lc_iid + n_lt_iid]
    + ood_hold_list[n_lc_ood : n_lc_ood + n_lt_ood]
)
latent_test_regimes = ["iid"] * n_lt_iid + ["ood"] * n_lt_ood

print()
print(f"latent_train_ids         : {len(latent_train_ids)}")
print(f"latent_calibration_ids   : {len(latent_calibration_ids)} "
      f"({n_lc_iid} iid + {n_lc_ood} ood)")
print(f"latent_test_ids          : {len(latent_test_ids)} "
      f"({n_lt_iid} iid + {n_lt_ood} ood)")

by_id = {p["row_id"]: p for p in pairs}
latent_train_rows       = [by_id[i] for i in latent_train_ids]
latent_calibration_rows = [by_id[i] for i in latent_calibration_ids]
latent_test_rows        = [by_id[i] for i in latent_test_ids]


In [ ]:
# Family-balanced weights

# When a template family has 2,000 rows and another has 30, unweighted metrics
# are dominated by the 2,000-row family. Compute weights w_i = 1 / n_{g(i)}
# and normalize; every downstream metric that could be biased by family
# concentration will be reported both row-weighted and family-balanced.

def family_balanced_weights(rows):
    counts = Counter(r["template_id"] for r in rows)
    w = np.array([1.0 / counts[r["template_id"]] for r in rows], dtype=np.float64)
    return w / w.sum() * len(rows)   # normalize so mean(w) = 1

w_train_fb = family_balanced_weights(latent_train_rows)
w_calib_fb = family_balanced_weights(latent_calibration_rows)
w_test_fb  = family_balanced_weights(latent_test_rows)

print(f"Family-balanced weights (train split, N={len(latent_train_rows)}):")
print(f"  min={w_train_fb.min():.4f}  median={np.median(w_train_fb):.4f}  max={w_train_fb.max():.4f}")
print(f"  effective sample size = (sum w)^2 / sum(w^2) "
      f"= {(w_train_fb.sum()**2 / (w_train_fb**2).sum()):.1f}")
print()
print(f"Family-balanced weights (calibration split, N={len(latent_calibration_rows)}):")
print(f"  effective sample size = {(w_calib_fb.sum()**2 / (w_calib_fb**2).sum()):.1f}")

In [ ]:
# Split assertions and split manifest

train_set = set(latent_train_ids)
calib_set = set(latent_calibration_ids)
test_set  = set(latent_test_ids)

# Disjointness at row level
assert train_set.isdisjoint(calib_set), "latent train ∩ calibration not empty"
assert train_set.isdisjoint(test_set),  "latent train ∩ test not empty"
assert calib_set.isdisjoint(test_set),  "latent calibration ∩ test not empty"

# Template-OOD guarantee: no template family in latent_train appears in latent_test's OOD portion
train_templates = {by_id[i]["template_id"] for i in latent_train_ids}
test_templates_ood = {by_id[i]["template_id"] for i in latent_test_ids if by_id[i]["template_id"] in ood_families}
assert train_templates.isdisjoint(test_templates_ood), "OOD template leaked into train"

# Front-of-file check: split must not degenerate into the first N rows
_max_train_idx = max(latent_train_ids)
assert _max_train_idx > len(pairs) // 4, (
    f"latent_train covers only front of dataset (max row_id={_max_train_idx} "
    f"but dataset size={len(pairs)}). Shuffling failed."
)

# Minimum family support: every calibration family has at least 1 calibration row
_calib_family_counts = Counter(by_id[i]["template_id"] for i in latent_calibration_ids)
assert all(c >= 1 for c in _calib_family_counts.values()), "some calibration family has 0 rows"

# Save split manifest
SPLIT_MANIFEST = {
    "config_hash": CONFIG_HASH,
    "master_seed": MASTER_SEED,
    "n_pairs_total": len(pairs),
    "n_train_pool": len(train_row_ids),
    "n_iid_holdout_pool": len(iid_holdout_row_ids),
    "n_ood_holdout_pool": len(ood_row_ids),
    "latent_train_row_ids": latent_train_ids,
    "latent_calibration_row_ids": latent_calibration_ids,
    "latent_test_row_ids": latent_test_ids,
    "latent_calibration_regimes": latent_calibration_regimes,
    "latent_test_regimes": latent_test_regimes,
    "ood_families_count": len(ood_families),
}
_split_path = MANIFEST_DIR / f"split_manifest_{CONFIG_HASH}.json"
_split_path.write_text(json.dumps(SPLIT_MANIFEST, indent=2))
print(f"All split assertions passed.")
print(f"Split manifest → {_split_path}")


## Find where the representation lives

Now we look at what Qwen actually sees. Two things matter before any pooling:

1. **View construction:** what exact string is being tokenized for each view? (Qwen's chat template silently injects a default system prompt if the first message isn't a system message. That's a hidden confound.)
2. **Content masks:** which tokens are actual SQL/prompt content, versus chat-template scaffold (`<|im_start|>`, role markers, `<|im_end|>`, trailing newlines)? Assuming "last token = SQL content" is exactly the assumption that torpedoed nb03's target representation.

In [ ]:
# Load tokenizer and inspect the rendered views

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

SHARED_SYSTEM = (
    "You are a PostgreSQL assistant for a commercial-insurance database. "
    "Return one SQL query. No prose, no code fences."
)

def render_prompt_view(prompt):
    """Chat-template render for the prompt view.

    We supply an EXPLICIT system message so Qwen does not silently inject its
    default. The assistant turn is added as a generation prompt (no target
    tokens included).
    """
    msgs = [
        {"role": "system", "content": SHARED_SYSTEM},
        {"role": "user",   "content": prompt},
    ]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def render_target_view(prompt, sql):
    """Chat-template render for the target view.

    Same explicit system and the same actual user prompt as the generation
    context, followed by the assistant SQL. The content mask still pools only
    SQL tokens from the assistant span, but those hidden states can attend to
    the prompt. That matches the JEPA question: can the prompt-side view
    predict the answer-side representation for this conversation?
    """
    msgs = [
        {"role": "system",    "content": SHARED_SYSTEM},
        {"role": "user",      "content": prompt},
        {"role": "assistant", "content": sql},
    ]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)

# Print rendered views for 3 examples
for i, pair in enumerate(latent_train_rows[:3]):
    print(f"── Example {i} (row_id={pair['row_id']}) ──")
    p_view = render_prompt_view(pair["prompt"])
    t_view = render_target_view(pair["prompt"], pair["target"])
    print(f"[PROMPT VIEW]  len={len(p_view)}")
    print(p_view)
    print(f"[TARGET VIEW]  len={len(t_view)}")
    print(t_view)
    print()


In [ ]:
# Create exact content masks

# Special tokens Qwen wraps around content
IM_START_ID = tokenizer.convert_tokens_to_ids("<|im_start|>")
IM_END_ID   = tokenizer.convert_tokens_to_ids("<|im_end|>")
EOT_ID      = tokenizer.eos_token_id
PAD_ID      = tokenizer.pad_token_id

# Role names Qwen tokenizes as single tokens for this local tokenizer.
ROLE_TOKEN_IDS = set()
ROLE_IDS = {}
for role in ("system", "user", "assistant"):
    ids = tokenizer(role, add_special_tokens=False)["input_ids"]
    if len(ids) == 1:
        ROLE_TOKEN_IDS.add(ids[0])
        ROLE_IDS[role] = ids[0]
    else:
        print(f"NOTE: role '{role}' tokenizes to {ids} (not a single token).")
        ROLE_IDS[role] = None

# The newline token that follows role markers
NEWLINE_IDS = tokenizer("\n", add_special_tokens=False)["input_ids"]
NEWLINE_ID = NEWLINE_IDS[0] if len(NEWLINE_IDS) == 1 else None

assert IM_START_ID is not None and IM_START_ID != tokenizer.unk_token_id, "Qwen <|im_start|> token not found"
assert IM_END_ID is not None and IM_END_ID != tokenizer.unk_token_id, "Qwen <|im_end|> token not found"
assert ROLE_IDS["user"] is not None and ROLE_IDS["assistant"] is not None, "Qwen role tokens must be single-token"
assert NEWLINE_ID is not None, "Qwen newline must be single-token for exact role-span masking"

SCAFFOLD_IDS = set()
for tid in (IM_START_ID, IM_END_ID, EOT_ID, PAD_ID, NEWLINE_ID):
    if tid is not None:
        SCAFFOLD_IDS.add(tid)
SCAFFOLD_IDS |= ROLE_TOKEN_IDS

def tokenize_with_mask(text, view_type):
    """Return input_ids, attention_mask, content_mask, terminal_idx, eot_idx, last_content_idx.

    content_mask[i] = True iff token i is actual prompt/SQL content (not scaffold).
    For the TARGET view, content is the assistant's SQL: tokens after the last
    <|im_start|>assistant\n and before the following <|im_end|>.
    For the PROMPT view, content is the user's prompt: tokens after the last
    <|im_start|>user\n and before the following <|im_end|>.

    terminal_idx      = last non-pad token index (what "last_token" pooling picks)
    eot_idx           = position of the FINAL <|im_end|> (or last non-pad if absent)
    last_content_idx  = last True index in content_mask
    """
    enc = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=SEQ_LEN,
        add_special_tokens=False,
    )
    ids = enc["input_ids"][0].tolist()
    attn = enc["attention_mask"][0].tolist()

    content_mask = [False] * len(ids)

    role = "assistant" if view_type == "target" else "user"
    role_id = ROLE_IDS[role]
    span_start = None
    for i in range(len(ids) - 2):
        if ids[i] == IM_START_ID and ids[i + 1] == role_id:
            span_start = i + 3 if (i + 2 < len(ids) and ids[i + 2] == NEWLINE_ID) else i + 2

    span_end = len(ids)
    if span_start is not None:
        for j in range(span_start, len(ids)):
            if ids[j] == IM_END_ID:
                span_end = j
                break
        for k in range(span_start, min(span_end, len(ids))):
            content_mask[k] = True

    terminal_idx = max(i for i, a in enumerate(attn) if a == 1)
    eot_positions = [i for i, t in enumerate(ids) if t == IM_END_ID]
    eot_idx = eot_positions[-1] if eot_positions else terminal_idx
    content_positions = [i for i, m in enumerate(content_mask) if m]
    last_content_idx = content_positions[-1] if content_positions else terminal_idx

    out = {
        "input_ids":        torch.tensor(ids),
        "attention_mask":   torch.tensor(attn),
        "content_mask":     torch.tensor(content_mask),
        "terminal_idx":     terminal_idx,
        "eot_idx":          eot_idx,
        "last_content_idx": last_content_idx,
    }
    assert_content_mask_invariants(out, view_type)
    return out

def assert_content_mask_invariants(tok, view_type):
    ids = tok["input_ids"].tolist()
    content = tok["content_mask"].tolist()
    content_positions = [i for i, yes in enumerate(content) if yes]

    assert content_positions, f"{view_type}: no content tokens found"
    assert not any(content[i] for i, t in enumerate(ids) if t == IM_END_ID), (
        f"{view_type}: im_end token included in content_mask"
    )
    if ids and ids[-1] == NEWLINE_ID:
        assert not content[-1], f"{view_type}: trailing newline included in content_mask"
    assert tok["last_content_idx"] == content_positions[-1], f"{view_type}: last_content_idx mismatch"
    assert tok["last_content_idx"] < tok["eot_idx"], f"{view_type}: content extends to or past final eot"

def print_mask_tail(text, view_type, row_id, tail=28):
    tok = tokenize_with_mask(text, view_type)
    print(f"{view_type.title()} view for row_id={row_id}")
    print(f"  seq_len = {len(tok['input_ids'])}, "
          f"terminal_idx = {tok['terminal_idx']}, "
          f"eot_idx = {tok['eot_idx']}, "
          f"last_content_idx = {tok['last_content_idx']}")
    print()
    print(f"{'idx':>4}  {'token_id':>8}  {'in_content':>10}  {'label':<18}  token_str")
    print("-" * 80)
    ids = tok["input_ids"].tolist()
    content = tok["content_mask"].tolist()
    labels = {}
    labels[tok["terminal_idx"]] = "terminal"
    labels[tok["eot_idx"]] = "eot" if tok["eot_idx"] != tok["terminal_idx"] else labels.get(tok["terminal_idx"], "") + "/eot"
    labels[tok["last_content_idx"]] = (labels.get(tok["last_content_idx"], "") + "/last_content").lstrip("/")

    for i in range(max(0, len(ids) - tail), len(ids)):
        tok_str = tokenizer.decode([ids[i]]).replace("\n", "\\n")
        yes = "Y" if content[i] else "N"
        lbl = labels.get(i, "")
        print(f"{i:>4}  {ids[i]:>8}  {yes:>10}  {lbl:<18}  {tok_str!r}")
    print()
    return tok

# Inspection: print token-by-token tables for prompt and target views.
_sample = latent_train_rows[0]
_prompt_tok = print_mask_tail(render_prompt_view(_sample["prompt"]), "prompt", _sample["row_id"])
_target_tok = print_mask_tail(render_target_view(_sample["prompt"], _sample["target"]), "target", _sample["row_id"])


In [ ]:
# Extract the latent panel

# We do one forward pass per (view, row) with output_hidden_states=True,
# pool at every (layer, pooling) combo, and cache to disk.
#
# Storage: 04-outputs/latent-cache/{split}_{CONFIG_HASH}.pt
#   dict keyed by (view, layer, pooling) -> float16 tensor (n_obs, D)
#   plus row_ids and cache metadata.

POOLING_FNS = {}

def _register(name):
    def deco(fn):
        POOLING_FNS[name] = fn
        return fn
    return deco

@_register("terminal")
def _pool_terminal(hidden, mask):
    idx = mask["terminal_idx"]
    return hidden[idx]

@_register("last_content")
def _pool_last_content(hidden, mask):
    idx = mask["last_content_idx"]
    return hidden[idx]

@_register("content_mean")
def _pool_content_mean(hidden, mask):
    cm = mask["content_mask"].to(hidden.device)
    if cm.sum() == 0:
        return hidden[mask["terminal_idx"]]
    return hidden[cm].mean(dim=0)

@_register("content_tail4")
def _pool_content_tail4(hidden, mask):
    cm = mask["content_mask"].to(hidden.device)
    if cm.sum() == 0:
        return hidden[mask["terminal_idx"]]
    idxs = torch.nonzero(cm, as_tuple=False).squeeze(-1)
    tail = idxs[-4:]
    return hidden[tail].mean(dim=0)

def _rows_cache_fingerprint(rows):
    h = hashlib.sha256()
    for r in rows:
        h.update(str(r["row_id"]).encode())
        h.update(b"\0")
        h.update(r["prompt"].encode("utf-8"))
        h.update(b"\0")
        h.update(r["target"].encode("utf-8"))
        h.update(b"\0")
    return h.hexdigest()[:16]

def _expected_cache_meta(split_name, rows):
    return {
        "config_hash": CONFIG_HASH,
        "split_name": split_name,
        "n_rows": len(rows),
        "rows_fingerprint": _rows_cache_fingerprint(rows),
        "view_contract_version": VIEW_CONTRACT_VERSION,
        "content_mask_version": CONTENT_MASK_VERSION,
        "extractor_version": EXTRACTOR_VERSION,
        "layers": LAYERS,
        "poolings": POOLINGS,
        "seq_len": SEQ_LEN,
        "model_name": MODEL_NAME,
    }

def load_model():
    print(f"Loading model from {MODEL_NAME}...")
    m = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=DTYPE)
    m.to(DEVICE)
    m.eval()
    for p in m.parameters():
        p.requires_grad_(False)
    print(f"  device={DEVICE}, dtype={DTYPE}, hidden_dim={m.config.hidden_size}")
    return m

def extract_split(rows, split_name):
    cache_path = CACHE_DIR / f"{split_name}_{CONFIG_HASH}.pt"
    expected_meta = _expected_cache_meta(split_name, rows)
    if cache_path.exists():
        cached = torch.load(cache_path, map_location="cpu")
        if cached.get("meta") == expected_meta:
            print(f"  Cache hit → {cache_path}")
            return cached
        raise RuntimeError(
            f"Stale cache metadata in {cache_path}. Delete it or rerun with a new CONFIG_HASH."
        )

    model = extract_split._model
    combos = [(v, ell, p) for v in ("prompt", "target") for ell in LAYERS for p in POOLINGS]
    buffers = {combo: [] for combo in combos}
    row_ids = []
    t0 = time.time()
    for i, row in enumerate(rows):
        row_ids.append(row["row_id"])
        for view in ("prompt", "target"):
            text = render_prompt_view(row["prompt"]) if view == "prompt" else render_target_view(row["prompt"], row["target"])
            mask = tokenize_with_mask(text, view)
            input_ids = mask["input_ids"].unsqueeze(0).to(DEVICE)
            attention_mask = mask["attention_mask"].unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                out = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    output_hidden_states=True,
                    use_cache=False,
                )
            hs = out.hidden_states  # tuple of (1, seq, D), length = n_layers + 1
            for ell in LAYERS:
                hidden = hs[ell][0]   # (seq, D)
                for pool_name in POOLINGS:
                    z = POOLING_FNS[pool_name](hidden, mask)
                    buffers[(view, ell, pool_name)].append(z.detach().to(torch.float16).cpu())
        if (i + 1) % 50 == 0 or i == len(rows) - 1:
            elapsed = time.time() - t0
            rate = (i + 1) / max(elapsed, 1e-9)
            eta = (len(rows) - (i + 1)) / max(rate, 1e-9)
            print(f"  {split_name}: {i+1}/{len(rows)}  "
                  f"({elapsed:.1f}s elapsed, ETA {eta/60:.1f} min, {rate:.2f} rows/s)")

    out_dict = {
        "row_ids": row_ids,
        "layers": LAYERS,
        "poolings": POOLINGS,
        "meta": expected_meta,
    }
    for combo, buf in buffers.items():
        out_dict[combo] = torch.stack(buf)   # (n_obs, D) fp16
    torch.save(out_dict, cache_path)
    print(f"  Saved → {cache_path}  ({cache_path.stat().st_size / 1e6:.1f} MB)")
    return out_dict

extract_split._model = None

# NOTE: this cell reserves the GPU. Only run it when the GPU is free.
# You can comment out the extract_split() calls and load prior caches instead.

if extract_split._model is None:
    extract_split._model = load_model()

panel_train = extract_split(latent_train_rows,       "train")
panel_calib = extract_split(latent_calibration_rows, "calibration")
panel_test  = extract_split(latent_test_rows,        "test")

# Free GPU memory once extraction is done; the rest of the notebook runs on CPU.
del extract_split._model
if DEVICE == "cuda":
    torch.cuda.empty_cache()
print()
print("Latent panels ready. Model unloaded. Downstream analysis runs on CPU.")


## Does a pair-specific signal exist?

Now that the latent panel is cached, everything below is CPU-only tensor math on ~800-1500 vectors of dim 896. Fast enough to iterate on interactively.

The four subsections ask:
1. **Geometry** — how anisotropic is each representation? What fraction of variance sits in a common direction? How much variance is between-family vs within-family?
2. **Constant baseline + retrieval** — does the ridge-predicted target beat the trivial "predict the mean target" baseline? Does the correct target rank above alternatives in cross-view retrieval?
3. **Frozen predictability** — fit `z_target = A · z_prompt + ε` on training rows, evaluate OOS on calibration. Does the aligned mapping generalize beyond the nb03 predictor's common-direction solution?
4. **Frisch-Waugh-Lovell decomposition** — residualize both views wrt template family. Once the family-mean is removed, does prompt still predict target's pair-specific residual?

Then the **selection gate** compiles these into raw and FWL pass/fail decisions for each `(layer, pooling)` candidate.


In [ ]:
# Geometry decomposition utilities

def _to_np32(x):
    return x.to(torch.float32).numpy() if isinstance(x, torch.Tensor) else x.astype(np.float32)

def anisotropy(Z):
    """Mean pairwise cosine among rows of Z. Near 1 = all rows point the same way."""
    Z = _to_np32(Z)
    Z_n = Z / (np.linalg.norm(Z, axis=1, keepdims=True) + 1e-12)
    N = Z_n.shape[0]
    # Sample pairs to avoid O(N^2) memory for large N
    if N > 400:
        idx = np.random.default_rng(MASTER_SEED).choice(N, 400, replace=False)
        Zs = Z_n[idx]
    else:
        Zs = Z_n
    S = Zs @ Zs.T
    iu = np.triu_indices_from(S, k=1)
    return float(S[iu].mean())

def effective_rank(Z):
    """Entropy-based effective rank of the covariance spectrum."""
    Z = _to_np32(Z)
    Z_c = Z - Z.mean(axis=0, keepdims=True)
    # SVD is more numerically stable than eig(cov) for tall thin matrices
    _, sv, _ = np.linalg.svd(Z_c, full_matrices=False)
    eigs = sv ** 2
    p = eigs / (eigs.sum() + 1e-12)
    p = p[p > 1e-12]
    H = -np.sum(p * np.log(p))
    return float(np.exp(H))

def common_direction_share(Z, K):
    """Fraction of variance in the top-K singular components."""
    Z = _to_np32(Z)
    Z_c = Z - Z.mean(axis=0, keepdims=True)
    _, sv, _ = np.linalg.svd(Z_c, full_matrices=False)
    eigs = sv ** 2
    if K >= len(eigs):
        return 1.0
    return float(eigs[:K].sum() / (eigs.sum() + 1e-12))

def template_icc(Z, groups):
    """Intraclass correlation: fraction of variance between groups.

    ICC = σ²_between / (σ²_between + σ²_within)
    Averaged across output dims of Z (896-dim).
    """
    Z = _to_np32(Z)
    groups = np.asarray(groups)
    global_mean = Z.mean(axis=0)
    between_ss = 0.0
    within_ss = 0.0
    counts = 0
    for g in np.unique(groups):
        idx = groups == g
        n_g = idx.sum()
        if n_g < 2:
            continue
        z_g = Z[idx]
        mean_g = z_g.mean(axis=0)
        between_ss += n_g * np.sum((mean_g - global_mean) ** 2)
        within_ss += np.sum((z_g - mean_g) ** 2)
        counts += n_g
    total = between_ss + within_ss
    if total < 1e-12:
        return 0.0
    return float(between_ss / total)

def constant_predictor_cos(Z_prompt, Z_target):
    """Mean cosine of the constant unconditional predictor (μ_T normalized) to each target."""
    Zp = _to_np32(Z_prompt)   # unused; the constant predictor ignores prompt
    Zt = _to_np32(Z_target)
    mu_T = Zt.mean(axis=0)
    mu_T_n = mu_T / (np.linalg.norm(mu_T) + 1e-12)
    Zt_n = Zt / (np.linalg.norm(Zt, axis=1, keepdims=True) + 1e-12)
    return float((Zt_n @ mu_T_n).mean())

def permutable_mask_for_groups(groups):
    """Rows in groups with at least two members can participate in within-family derangements."""
    groups = np.asarray(groups)
    counts = Counter(groups.tolist())
    mask = np.array([counts[g] >= 2 for g in groups], dtype=bool)
    singleton_excluded = int((~mask).sum())
    return mask, singleton_excluded

def _derange_indices(idx, rng, max_tries=64):
    idx = np.asarray(idx)
    if len(idx) < 2:
        return idx.copy()
    for _ in range(max_tries):
        shuf = rng.permutation(idx)
        if not np.any(shuf == idx):
            return shuf
    return np.roll(idx, 1)

def aligned_shuffled_cos(Z_prompt, Z_target, groups=None, mode="all", rng=None):
    """Return (aligned_mean, shuffled_mean).

    mode='all'          : shuffled = uniform random derangement over N
    mode='within_family': shuffled = derangement within same template family;
                          singleton_excluded rows are excluded, never left aligned
    mode='across_family': shuffled = random target from a different family
    """
    Zp = _to_np32(Z_prompt); Zt = _to_np32(Z_target)
    N = Zp.shape[0]
    Zp_n = Zp / (np.linalg.norm(Zp, axis=1, keepdims=True) + 1e-12)
    Zt_n = Zt / (np.linalg.norm(Zt, axis=1, keepdims=True) + 1e-12)

    if rng is None:
        rng = np.random.default_rng(MASTER_SEED)

    perm = np.arange(N)
    eval_idx = np.arange(N)
    if mode == "all":
        perm = _derange_indices(np.arange(N), rng)
    elif mode == "within_family":
        groups = np.asarray(groups)
        keep, singleton_excluded = permutable_mask_for_groups(groups)
        eval_idx = np.where(keep)[0]
        if len(eval_idx) == 0:
            return float("nan"), float("nan")
        for g in np.unique(groups[keep]):
            idx = np.where(groups == g)[0]
            perm[idx] = _derange_indices(idx, rng)
    elif mode == "across_family":
        groups = np.asarray(groups)
        for i in range(N):
            candidates = np.where(groups != groups[i])[0]
            if len(candidates) > 0:
                perm[i] = rng.choice(candidates)
    else:
        raise ValueError(mode)

    aligned = float((Zp_n[eval_idx] * Zt_n[eval_idx]).sum(axis=1).mean())
    shuffled = float((Zp_n[eval_idx] * Zt_n[perm[eval_idx]]).sum(axis=1).mean())
    return aligned, shuffled

print("Geometry utilities loaded.")


In [ ]:
# Cross-view retrieval

def retrieval_metrics(Z_prompt, Z_target, groups=None):
    """Cross-view retrieval on the aligned diagonal.

    Returns dict of Recall@{1,5}, MRR, median rank of correct target,
    same-family hard-negative rank, cross-family easy-negative rank.
    """
    Zp = _to_np32(Z_prompt); Zt = _to_np32(Z_target)
    N = Zp.shape[0]
    Zp_n = Zp / (np.linalg.norm(Zp, axis=1, keepdims=True) + 1e-12)
    Zt_n = Zt / (np.linalg.norm(Zt, axis=1, keepdims=True) + 1e-12)
    S = Zp_n @ Zt_n.T   # (N, N)

    # Rank of correct target (higher cos = better rank; rank 0 = top)
    diag = np.diag(S)
    ranks = np.zeros(N, dtype=int)
    for i in range(N):
        row = S[i]
        # rank of the diagonal element among row entries (descending)
        higher = np.sum(row > diag[i])
        ranks[i] = higher   # 0-indexed: 0 = top

    recall_at_1 = float((ranks == 0).mean())
    recall_at_5 = float((ranks < 5).mean())
    mrr = float((1.0 / (ranks + 1)).mean())
    med_rank = float(np.median(ranks))

    out = {
        "recall@1": recall_at_1,
        "recall@5": recall_at_5,
        "mrr":      mrr,
        "median_rank": med_rank,
    }

    if groups is not None:
        groups = np.asarray(groups)
        # For each row, split negatives into same-family vs cross-family
        same_ranks = []
        cross_ranks = []
        for i in range(N):
            row = S[i].copy()
            row[i] = -np.inf   # exclude diagonal
            same_mask = (groups == groups[i]) & (np.arange(N) != i)
            cross_mask = groups != groups[i]
            if same_mask.sum() > 0:
                # rank of top same-family negative among all negatives
                sorted_desc = np.argsort(-row)
                for r, idx in enumerate(sorted_desc):
                    if same_mask[idx]:
                        same_ranks.append(r)
                        break
            if cross_mask.sum() > 0:
                sorted_desc = np.argsort(-row)
                for r, idx in enumerate(sorted_desc):
                    if cross_mask[idx]:
                        cross_ranks.append(r)
                        break
        out["same_family_top_neg_median_rank"] = float(np.median(same_ranks)) if same_ranks else None
        out["cross_family_top_neg_median_rank"] = float(np.median(cross_ranks)) if cross_ranks else None

    return out

print("Retrieval utilities loaded.")

In [ ]:
# Frozen linear predictability + permutation nulls

def ridge_fit(X, Y, alpha):
    """Closed-form ridge: A = (X^T X + α I)^-1 X^T Y."""
    D = X.shape[1]
    XtX = X.T @ X
    reg = alpha * np.eye(D, dtype=X.dtype)
    A = np.linalg.solve(XtX + reg, X.T @ Y)
    return A

def r2_score(Y_true, Y_pred):
    """R² averaged across output dims. Positive = better than the mean predictor."""
    Y_true = _to_np32(Y_true); Y_pred = _to_np32(Y_pred)
    mean_Y = Y_true.mean(axis=0, keepdims=True)
    ss_res = np.sum((Y_true - Y_pred) ** 2)
    ss_tot = np.sum((Y_true - mean_Y) ** 2)
    if ss_tot < 1e-12:
        return 0.0
    return float(1.0 - ss_res / ss_tot)

def cv_ridge_r2(X_train, Y_train, X_test, Y_test, alphas, val_frac=0.2, rng=None):
    """Fit ridge on X_train, pick α on internal val, evaluate on X_test.

    Returns (best_alpha, r2_test, fitted_A).
    """
    X_train = _to_np32(X_train); Y_train = _to_np32(Y_train)
    X_test = _to_np32(X_test);   Y_test = _to_np32(Y_test)
    N = X_train.shape[0]
    if rng is None:
        rng = np.random.default_rng(MASTER_SEED)
    idx = rng.permutation(N)
    n_val = int(N * val_frac)
    val_idx = idx[:n_val]
    fit_idx = idx[n_val:]
    Xf, Yf = X_train[fit_idx], Y_train[fit_idx]
    Xv, Yv = X_train[val_idx], Y_train[val_idx]

    best_alpha = None
    best_val_r2 = -np.inf
    best_A = None
    for a in alphas:
        A = ridge_fit(Xf, Yf, a)
        val_pred = Xv @ A
        r2 = r2_score(Yv, val_pred)
        if r2 > best_val_r2:
            best_val_r2 = r2
            best_alpha = a
            best_A = A

    test_pred = X_test @ best_A
    r2_test = r2_score(Y_test, test_pred)
    return float(best_alpha), r2_test, best_A

def permutation_null_r2(X_train, Y_train, X_test, Y_test, alpha, n_perm=250,
                        mode="all", train_groups=None, test_groups=None, rng=None):
    """Distribution of test R² under target-shuffling.

    mode='all'          : shuffle Y_train globally (fixed-point-free)
    mode='within_family': shuffle Y_train within each training family; singleton
                          groups are excluded from the null fit, not left aligned
    mode='across_family': shuffle Y_train across families only
    """
    X_train = _to_np32(X_train); Y_train = _to_np32(Y_train)
    X_test = _to_np32(X_test); Y_test = _to_np32(Y_test)
    if rng is None:
        rng = np.random.default_rng(MASTER_SEED + 7)

    if mode == "within_family":
        train_groups = np.asarray(train_groups)
        keep, singleton_excluded = permutable_mask_for_groups(train_groups)
        X_fit = X_train[keep]
        Y_fit = Y_train[keep]
        groups_fit = train_groups[keep]
        if X_fit.shape[0] < 2:
            return np.full(n_perm, np.nan)
    else:
        X_fit = X_train
        Y_fit = Y_train
        groups_fit = np.asarray(train_groups) if train_groups is not None else None

    N = X_fit.shape[0]
    D = X_fit.shape[1]
    XtX = X_fit.T @ X_fit
    W = np.linalg.solve(XtX + alpha * np.eye(D, dtype=X_fit.dtype), X_fit.T)

    r2s = np.empty(n_perm)
    for t in range(n_perm):
        if mode == "all":
            perm = _derange_indices(np.arange(N), rng)
        elif mode == "within_family":
            perm = np.arange(N)
            for g in np.unique(groups_fit):
                idx = np.where(groups_fit == g)[0]
                perm[idx] = _derange_indices(idx, rng)
        elif mode == "across_family":
            groups_fit = np.asarray(groups_fit)
            perm = np.arange(N)
            for i in range(N):
                candidates = np.where(groups_fit != groups_fit[i])[0]
                if len(candidates) > 0:
                    perm[i] = rng.choice(candidates)
        else:
            raise ValueError(mode)

        A_perm = W @ Y_fit[perm]
        pred = X_test @ A_perm
        r2s[t] = r2_score(Y_test, pred)
    return r2s

print("Ridge + permutation-null utilities loaded.")


In [ ]:
# Frisch-Waugh-Lovell residualization

def residualize_by_group(Z, groups):
    """Return Z̃ = Z - E[Z | group], i.e. row-wise deviation from family mean.

    This is the projection that removes the α_{g(i)} component from the
    z_T = μ + α_g + s_i decomposition. What remains is (approximately)
    the pair-specific residual + noise.
    """
    Z = _to_np32(Z)
    groups = np.asarray(groups)
    Z_out = Z.copy()
    for g in np.unique(groups):
        idx = groups == g
        if idx.sum() == 0:
            continue
        mean_g = Z[idx].mean(axis=0, keepdims=True)
        Z_out[idx] = Z[idx] - mean_g
    # Also remove global mean (μ)
    Z_out = Z_out - Z_out.mean(axis=0, keepdims=True)
    return Z_out


def run_utility_sanity_checks():
    """Cheap invariants for utilities that feed the selection gate."""
    D = 6
    Z = np.eye(D, dtype=np.float32)
    retr = retrieval_metrics(Z, Z)  # retrieval_metrics(Z, Z) -> perfect diagonal retrieval
    assert retr["recall@1"] == 1.0 and retr["mrr"] == 1.0, retr

    Z_rank = np.vstack([np.eye(D), -np.eye(D)]).astype(np.float32)
    er = effective_rank(Z_rank)
    assert abs(er - D) < 0.25, f"effective_rank sanity failed: {er} vs {D}"

    iid_data = np.array([[1.0, 0.0], [-1.0, 0.0], [0.0, 1.0], [0.0, -1.0]], dtype=np.float32)
    random_groups = np.array([0, 0, 1, 1])
    icc = template_icc(iid_data, random_groups)  # template_icc(iid_data, random_groups) ≈ 0
    assert abs(icc) < 1e-8, f"template_icc sanity failed: {icc}"

    resid = residualize_by_group(iid_data, random_groups)
    for g in np.unique(random_groups):
        assert np.allclose(resid[random_groups == g].mean(axis=0), 0.0, atol=1e-7)
    assert np.allclose(resid.mean(axis=0), 0.0, atol=1e-7)

run_utility_sanity_checks()
print("FWL residualization loaded; utility sanity checks passed.")


In [ ]:
# Evaluate every (layer, pooling) candidate

# For each combo, produce a metrics dict. All computations use training rows
# ONLY for fitting; calibration rows for evaluation. TEST ROWS ARE UNTOUCHED
# until the training notebook has selected a winner.

train_groups = np.array([r["template_id"] for r in latent_train_rows])
calib_groups = np.array([r["template_id"] for r in latent_calibration_rows])

# For retrieval metrics, we need groups labeled numerically for the "same family"
# distinctions to work cleanly.
all_group_names = sorted(set(train_groups) | set(calib_groups))
_group_to_id = {g: i for i, g in enumerate(all_group_names)}
train_group_ids = np.array([_group_to_id[g] for g in train_groups])
calib_group_ids = np.array([_group_to_id[g] for g in calib_groups])
calib_within_keep, calib_singleton_excluded = permutable_mask_for_groups(calib_group_ids)
train_within_keep, train_singleton_excluded = permutable_mask_for_groups(train_group_ids)
calib_regimes = np.array(latent_calibration_regimes)
calib_iid_idx = np.where(calib_regimes == "iid")[0]
calib_ood_idx = np.where(calib_regimes == "ood")[0]

RESULTS = []   # one row per (layer, pooling)

for ell in LAYERS:
    for pool_name in POOLINGS:
        print(f"── Layer {ell}, pooling '{pool_name}' ──")
        Zp_tr = panel_train[("prompt", ell, pool_name)]
        Zt_tr = panel_train[("target", ell, pool_name)]
        Zp_ca = panel_calib[("prompt", ell, pool_name)]
        Zt_ca = panel_calib[("target", ell, pool_name)]

        Zp_tr_np = _to_np32(Zp_tr); Zt_tr_np = _to_np32(Zt_tr)
        Zp_ca_np = _to_np32(Zp_ca); Zt_ca_np = _to_np32(Zt_ca)

        # ─── Geometry ─────────────────────────────────────────────────────
        geom = {
            "target_anisotropy":      anisotropy(Zt_tr_np),
            "prompt_anisotropy":      anisotropy(Zp_tr_np),
            "target_effective_rank":  effective_rank(Zt_tr_np),
            "prompt_effective_rank":  effective_rank(Zp_tr_np),
            "target_top1_var_share":  common_direction_share(Zt_tr_np, 1),
            "target_top4_var_share":  common_direction_share(Zt_tr_np, 4),
            "target_top8_var_share":  common_direction_share(Zt_tr_np, 8),
            "target_template_icc":    template_icc(Zt_tr_np, train_group_ids),
            "prompt_template_icc":    template_icc(Zp_tr_np, train_group_ids),
        }

        # ─── Constant-predictor baseline ─────────────────────────────────
        # Fit μ_T on training rows, evaluate on calibration.
        mu_T = Zt_tr_np.mean(axis=0)
        mu_T_n = mu_T / (np.linalg.norm(mu_T) + 1e-12)
        Zt_ca_n = Zt_ca_np / (np.linalg.norm(Zt_ca_np, axis=1, keepdims=True) + 1e-12)
        const_pred_cos = float((Zt_ca_n @ mu_T_n).mean())

        # ─── Raw aligned / shuffled cosine (diagnostic only) ──────────────
        raw_aligned_cos_all,    raw_shuffled_cos_all    = aligned_shuffled_cos(Zp_ca_np, Zt_ca_np, mode="all", groups=calib_group_ids, rng=np.random.default_rng(MASTER_SEED))
        raw_aligned_cos_within, raw_shuffled_cos_within = aligned_shuffled_cos(Zp_ca_np, Zt_ca_np, groups=calib_group_ids, mode="within_family", rng=np.random.default_rng(MASTER_SEED + 1))
        raw_aligned_cos_across, raw_shuffled_cos_across = aligned_shuffled_cos(Zp_ca_np, Zt_ca_np, groups=calib_group_ids, mode="across_family", rng=np.random.default_rng(MASTER_SEED + 2))
        raw_retr = retrieval_metrics(Zp_ca_np, Zt_ca_np, groups=calib_group_ids)

        # ─── Frozen ridge OOS (train → calibration) ──────────────────────
        best_alpha, r2_aligned, A_star = cv_ridge_r2(
            Zp_tr_np, Zt_tr_np, Zp_ca_np, Zt_ca_np, RIDGE_ALPHAS,
            val_frac=1 - TRAIN_VAL_SPLIT, rng=np.random.default_rng(MASTER_SEED + 3),
        )
        ridge_pred_calib = Zp_ca_np @ A_star
        r2_aligned_iid = r2_score(Zt_ca_np[calib_iid_idx], ridge_pred_calib[calib_iid_idx]) if len(calib_iid_idx) else None
        r2_aligned_ood = r2_score(Zt_ca_np[calib_ood_idx], ridge_pred_calib[calib_ood_idx]) if len(calib_ood_idx) else None

        ridge_pred_cos_all,    ridge_pred_shuffled_cos_all    = aligned_shuffled_cos(ridge_pred_calib, Zt_ca_np, mode="all", groups=calib_group_ids, rng=np.random.default_rng(MASTER_SEED + 30))
        ridge_pred_cos_within, ridge_pred_shuffled_cos_within = aligned_shuffled_cos(ridge_pred_calib, Zt_ca_np, groups=calib_group_ids, mode="within_family", rng=np.random.default_rng(MASTER_SEED + 31))
        ridge_pred_cos_across, ridge_pred_shuffled_cos_across = aligned_shuffled_cos(ridge_pred_calib, Zt_ca_np, groups=calib_group_ids, mode="across_family", rng=np.random.default_rng(MASTER_SEED + 32))
        ridge_pred_retrieval = retrieval_metrics(ridge_pred_calib, Zt_ca_np, groups=calib_group_ids)
        ridge_pred_const_advantage = ridge_pred_cos_all - const_pred_cos

        # ─── Permutation nulls (test R² distributions) ───────────────────
        null_all      = permutation_null_r2(Zp_tr_np, Zt_tr_np, Zp_ca_np, Zt_ca_np, best_alpha,
                                            n_perm=N_PERMUTATIONS, mode="all",
                                            train_groups=train_group_ids,
                                            rng=np.random.default_rng(MASTER_SEED + 10))
        null_within   = permutation_null_r2(Zp_tr_np, Zt_tr_np, Zp_ca_np, Zt_ca_np, best_alpha,
                                            n_perm=N_PERMUTATIONS, mode="within_family",
                                            train_groups=train_group_ids,
                                            rng=np.random.default_rng(MASTER_SEED + 11))
        null_across   = permutation_null_r2(Zp_tr_np, Zt_tr_np, Zp_ca_np, Zt_ca_np, best_alpha,
                                            n_perm=N_PERMUTATIONS, mode="across_family",
                                            train_groups=train_group_ids,
                                            rng=np.random.default_rng(MASTER_SEED + 12))

        pctile_all    = float((null_all    < r2_aligned).mean() * 100)
        pctile_within = float((null_within < r2_aligned).mean() * 100)
        pctile_across = float((null_across < r2_aligned).mean() * 100)

        # ─── FWL residualization (family partialled out) ─────────────────
        Zp_tr_res = residualize_by_group(Zp_tr_np, train_group_ids)
        Zt_tr_res = residualize_by_group(Zt_tr_np, train_group_ids)
        Zp_ca_res = residualize_by_group(Zp_ca_np, calib_group_ids)
        Zt_ca_res = residualize_by_group(Zt_ca_np, calib_group_ids)
        best_alpha_res, r2_residual, A_res = cv_ridge_r2(
            Zp_tr_res, Zt_tr_res, Zp_ca_res, Zt_ca_res, RIDGE_ALPHAS,
            val_frac=1 - TRAIN_VAL_SPLIT, rng=np.random.default_rng(MASTER_SEED + 4),
        )
        ridge_pred_res_calib = Zp_ca_res @ A_res
        r2_residual_iid = r2_score(Zt_ca_res[calib_iid_idx], ridge_pred_res_calib[calib_iid_idx]) if len(calib_iid_idx) else None
        r2_residual_ood = r2_score(Zt_ca_res[calib_ood_idx], ridge_pred_res_calib[calib_ood_idx]) if len(calib_ood_idx) else None
        null_res_all = permutation_null_r2(
            Zp_tr_res, Zt_tr_res, Zp_ca_res, Zt_ca_res, best_alpha_res,
            n_perm=N_PERMUTATIONS, mode="all",
            train_groups=train_group_ids,
            rng=np.random.default_rng(MASTER_SEED + 40),
        )
        pctile_res_all = float((null_res_all < r2_residual).mean() * 100)

        combo_result = {
            "layer": ell,
            "pooling": pool_name,
            "geometry": geom,
            "singleton_excluded": {
                "train_within_family": int(train_singleton_excluded),
                "calibration_within_family": int(calib_singleton_excluded),
            },
            "const_pred_cos": const_pred_cos,
            "raw_const_advantage_all": raw_aligned_cos_all - const_pred_cos,
            "raw_aligned_cos_all":     raw_aligned_cos_all,
            "raw_shuffled_cos_all":    raw_shuffled_cos_all,
            "raw_aligned_cos_within":  raw_aligned_cos_within,
            "raw_shuffled_cos_within": raw_shuffled_cos_within,
            "raw_aligned_cos_across":  raw_aligned_cos_across,
            "raw_shuffled_cos_across": raw_shuffled_cos_across,
            "raw_retrieval": raw_retr,
            "ridge_best_alpha":     best_alpha,
            "r2_aligned_calib":     r2_aligned,
            "r2_aligned_calib_iid": r2_aligned_iid,
            "r2_aligned_calib_ood": r2_aligned_ood,
            "ridge_pred_cos_all":     ridge_pred_cos_all,
            "ridge_pred_shuffled_cos_all": ridge_pred_shuffled_cos_all,
            "ridge_pred_cos_within":  ridge_pred_cos_within,
            "ridge_pred_shuffled_cos_within": ridge_pred_shuffled_cos_within,
            "ridge_pred_cos_across":  ridge_pred_cos_across,
            "ridge_pred_shuffled_cos_across": ridge_pred_shuffled_cos_across,
            "ridge_pred_retrieval": ridge_pred_retrieval,
            "ridge_pred_const_advantage": ridge_pred_const_advantage,
            "const_pred_advantage_all": ridge_pred_const_advantage,  # backward-compatible alias
            "r2_null_all_pctile":     pctile_all,
            "r2_null_within_pctile":  pctile_within,
            "r2_null_across_pctile":  pctile_across,
            "r2_null_all_p95":        float(np.nanpercentile(null_all, 95)),
            "r2_null_within_p95":     float(np.nanpercentile(null_within, 95)),
            "r2_null_across_p95":     float(np.nanpercentile(null_across, 95)),
            "fwl_ridge_best_alpha":   best_alpha_res,
            "r2_residualized_calib":  r2_residual,
            "r2_residualized_calib_iid": r2_residual_iid,
            "r2_residualized_calib_ood": r2_residual_ood,
            "r2_residual_null_all_pctile": pctile_res_all,
            "r2_residual_null_all_p95": float(np.nanpercentile(null_res_all, 95)),
        }
        RESULTS.append(combo_result)
        print(f"  ridge R²     = {r2_aligned:+.4f}   (α*={best_alpha:.2e})")
        print(f"  null@95(all) = {combo_result['r2_null_all_p95']:+.4f}   → aligned at {pctile_all:.1f} pctile")
        print(f"  ridge R² iid/ood = {r2_aligned_iid:+.4f} / {r2_aligned_ood:+.4f}")
        print(f"  pred MRR     = {ridge_pred_retrieval['mrr']:.4f}   R@1={ridge_pred_retrieval['recall@1']:.4f}")
        print(f"  pred const adv = {ridge_pred_const_advantage:+.4f}   "
              f"target eff-rank = {geom['target_effective_rank']:.1f}")
        print(f"  residualized R² = {r2_residual:+.4f}   "
              f"resid null pctile = {pctile_res_all:.1f}   template ICC (target) = {geom['target_template_icc']:.3f}")
        print()

# Save all results
_results_path = GEOMETRY_DIR / f"candidate_table_{CONFIG_HASH}.json"
_results_path.write_text(json.dumps(RESULTS, indent=2))
print(f"Saved → {_results_path}")


In [ ]:
# Summary table (calibration set only)

def _fmt_metric(x):
    return "n/a" if x is None else f"{x:+.4f}"

print(f"{'layer':>5}  {'pooling':<16}  {'R²_all':>9}  {'R²_iid':>9}  {'R²_ood':>9}  {'null_p95':>10}  {'pctile':>7}  "
      f"{'pred_MRR':>8}  {'pred_R@1':>8}  {'pred_adv':>10}  {'FWL_all':>8}  {'FWL_iid':>8}  {'FWL_ood':>8}  {'FWL_pct':>7}  {'eff_rank':>9}  {'ICC_T':>6}")
print("-" * 166)
for r in sorted(RESULTS, key=lambda x: -x["r2_aligned_calib"]):
    print(f"{r['layer']:>5}  {r['pooling']:<16}  "
          f"{r['r2_aligned_calib']:>+9.4f}  "
          f"{_fmt_metric(r['r2_aligned_calib_iid']):>9}  "
          f"{_fmt_metric(r['r2_aligned_calib_ood']):>9}  "
          f"{r['r2_null_all_p95']:>+10.4f}  "
          f"{r['r2_null_all_pctile']:>7.1f}  "
          f"{r['ridge_pred_retrieval']['mrr']:>8.4f}  "
          f"{r['ridge_pred_retrieval']['recall@1']:>8.4f}  "
          f"{r['ridge_pred_const_advantage']:>+10.4f}  "
          f"{r['r2_residualized_calib']:>+8.4f}  "
          f"{_fmt_metric(r['r2_residualized_calib_iid']):>8}  "
          f"{_fmt_metric(r['r2_residualized_calib_ood']):>8}  "
          f"{r['r2_residual_null_all_pctile']:>7.1f}  "
          f"{r['geometry']['target_effective_rank']:>9.1f}  "
          f"{r['geometry']['target_template_icc']:>6.3f}")


In [ ]:
# Representation selection gate

# The candidate representation is chosen on CALIBRATION data only. If no
# candidate passes either raw or FWL gates, the notebook stops and concludes
# that no usable prompt→target representation was identified.
#
# Baseline for the effective-rank ratio: the "terminal" pooling at layer -1
# (nb03's default). Every raw candidate, including the baseline itself,
# is judged by the same ratio criterion.

terminal_baseline_result = next(r for r in RESULTS if r["layer"] == -1 and r["pooling"] == "terminal")
terminal_eff_rank = terminal_baseline_result["geometry"]["target_effective_rank"]

def gate_check_raw(r):
    reasons = []
    beats_null   = r["r2_null_all_pctile"] >= GATE_ALIGNED_ABOVE_NULL_PCTILE
    beats_mrr    = r["ridge_pred_retrieval"]["mrr"] >= GATE_MIN_RETRIEVAL_MRR
    beats_const  = r["ridge_pred_const_advantage"] >= GATE_MIN_CONSTANT_PRED_ADVANTAGE
    beats_rank   = r["geometry"]["target_effective_rank"] / max(terminal_eff_rank, 1e-6) >= GATE_MIN_EFFECTIVE_RANK_RATIO

    if not beats_null:  reasons.append(f"ridge R² only at {r['r2_null_all_pctile']:.1f} pctile of null (need ≥{GATE_ALIGNED_ABOVE_NULL_PCTILE})")
    if not beats_mrr:   reasons.append(f"predicted-target MRR {r['ridge_pred_retrieval']['mrr']:.4f} < {GATE_MIN_RETRIEVAL_MRR}")
    if not beats_const: reasons.append(f"predicted const advantage {r['ridge_pred_const_advantage']:+.4f} < {GATE_MIN_CONSTANT_PRED_ADVANTAGE}")
    if not beats_rank:  reasons.append(f"effective-rank ratio {r['geometry']['target_effective_rank']/terminal_eff_rank:.2f} < {GATE_MIN_EFFECTIVE_RANK_RATIO}")

    return (beats_null and beats_mrr and beats_const and beats_rank), reasons

def gate_check_fwl(r):
    reasons = []
    beats_null = r["r2_residual_null_all_pctile"] >= GATE_ALIGNED_ABOVE_NULL_PCTILE
    positive = r["r2_residualized_calib"] > max(0.0, r["r2_residual_null_all_p95"])

    if not beats_null:
        reasons.append(f"residualized R² only at {r['r2_residual_null_all_pctile']:.1f} pctile of null (need ≥{GATE_ALIGNED_ABOVE_NULL_PCTILE})")
    if not positive:
        reasons.append(f"residualized R² {r['r2_residualized_calib']:+.4f} does not beat max(0, null p95={r['r2_residual_null_all_p95']:+.4f})")

    return (beats_null and positive), reasons

def select_candidate(candidates, score_key, p95_key):
    candidates_sorted = sorted(candidates, key=lambda x: -x[score_key])
    leader = candidates_sorted[0]
    null_std = np.std(np.array([r[p95_key] for r in candidates_sorted]))
    within_1se = [r for r in candidates_sorted if r[score_key] >= leader[score_key] - max(null_std, 0.01)]

    _pool_cost = {"terminal": 0, "last_content": 1, "content_tail4": 2, "content_mean": 3}
    _layer_cost = lambda ell: -ell   # -1 is cheapest, -8 needs deeper caching if separate pass

    if GATE_ONE_SE_SELECTION and len(within_1se) > 1:
        chosen = min(within_1se, key=lambda r: (_layer_cost(r["layer"]), _pool_cost[r["pooling"]]))
        note = f"1-SE rule applied ({len(within_1se)} candidates within SE of leader; picked cheapest)"
    else:
        chosen = leader
        note = "single leader"
    return chosen, note

raw_passing = []
fwl_passing = []
failing = []

for r in RESULTS:
    raw_ok, raw_reasons = gate_check_raw(r)
    fwl_ok, fwl_reasons = gate_check_fwl(r)
    r["raw_gate_passed"] = raw_ok
    r["fwl_gate_passed"] = fwl_ok
    r["raw_gate_failure_reasons"] = raw_reasons
    r["fwl_gate_failure_reasons"] = fwl_reasons
    r["gate_passed"] = raw_ok or fwl_ok
    r["gate_failure_reasons"] = raw_reasons + fwl_reasons
    if raw_ok:
        raw_passing.append(r)
    if fwl_ok:
        fwl_passing.append(r)
    if not (raw_ok or fwl_ok):
        failing.append(r)

if raw_passing and fwl_passing:
    gate_branch = "raw_and_residual"
elif raw_passing:
    gate_branch = "raw_jepa"
elif fwl_passing:
    gate_branch = "residual_jepa"
else:
    gate_branch = "none"

print(f"Gate results: raw={len(raw_passing)} passing, FWL={len(fwl_passing)} passing, total={len(RESULTS)}")
print(f"Branch decision: {gate_branch}")
print()

CHOSEN_RAW = None
CHOSEN_FWL = None
RAW_SELECTION_NOTE = None
FWL_SELECTION_NOTE = None

if raw_passing:
    CHOSEN_RAW, RAW_SELECTION_NOTE = select_candidate(raw_passing, "r2_aligned_calib", "r2_null_all_p95")
if fwl_passing:
    CHOSEN_FWL, FWL_SELECTION_NOTE = select_candidate(fwl_passing, "r2_residualized_calib", "r2_residual_null_all_p95")

if gate_branch == "none":
    print("═" * 70)
    print("NO CANDIDATE PASSED THE SELECTION GATE.")
    print("═" * 70)
    print()
    print("Reasons per candidate:")
    for r in sorted(failing, key=lambda x: max(x["r2_aligned_calib"], x["r2_residualized_calib"]), reverse=True)[:6]:
        print(f"  [{r['layer']:+d}, {r['pooling']:<16}]  raw R²={r['r2_aligned_calib']:+.4f}, FWL R²={r['r2_residualized_calib']:+.4f}")
        print("      raw:")
        for reason in r["raw_gate_failure_reasons"]:
            print(f"        - {reason}")
        print("      FWL:")
        for reason in r["fwl_gate_failure_reasons"]:
            print(f"        - {reason}")
    print()
    print("Interpretation: no (layer, pooling) on base Qwen 0.5B exposes")
    print("enough pair-specific signal on this data to justify a JEPA")
    print("training run. Next diagnostic step: rerun the preflight on the")
    print("SFT-trained checkpoint (from nb01/03) to see if fine-tuning")
    print("has reshaped the target manifold. If that also fails,")
    print("the diagnosis is that the data itself is not a viable substrate.")
elif gate_branch == "raw_jepa":
    print("═" * 70)
    print(f"RAW JEPA CANDIDATE: layer={CHOSEN_RAW['layer']}, pooling='{CHOSEN_RAW['pooling']}'")
    print("═" * 70)
    print(f"  Ridge R² (calib)  : {CHOSEN_RAW['r2_aligned_calib']:+.4f}")
    print(f"  vs null p95       : {CHOSEN_RAW['r2_null_all_p95']:+.4f}   (at {CHOSEN_RAW['r2_null_all_pctile']:.1f} pctile)")
    print(f"  Pred MRR          : {CHOSEN_RAW['ridge_pred_retrieval']['mrr']:.4f}   R@1={CHOSEN_RAW['ridge_pred_retrieval']['recall@1']:.4f}")
    print(f"  Pred const adv    : {CHOSEN_RAW['ridge_pred_const_advantage']:+.4f}")
    print(f"  Target effective rank: {CHOSEN_RAW['geometry']['target_effective_rank']:.1f}")
    print(f"  Selection note    : {RAW_SELECTION_NOTE}")
elif gate_branch == "residual_jepa":
    print("═" * 70)
    print(f"RESIDUAL JEPA CANDIDATE: layer={CHOSEN_FWL['layer']}, pooling='{CHOSEN_FWL['pooling']}'")
    print("═" * 70)
    print(f"  FWL residualized R²: {CHOSEN_FWL['r2_residualized_calib']:+.4f}")
    print(f"  vs residual null p95: {CHOSEN_FWL['r2_residual_null_all_p95']:+.4f}   (at {CHOSEN_FWL['r2_residual_null_all_pctile']:.1f} pctile)")
    print(f"  Raw ridge R²       : {CHOSEN_FWL['r2_aligned_calib']:+.4f}")
    print(f"  Selection note     : {FWL_SELECTION_NOTE}")
else:
    print("═" * 70)
    print("RAW AND RESIDUAL CANDIDATES PASSED.")
    print("═" * 70)
    print(f"  Raw candidate      : layer={CHOSEN_RAW['layer']}, pooling='{CHOSEN_RAW['pooling']}', R²={CHOSEN_RAW['r2_aligned_calib']:+.4f}")
    print(f"  Residual candidate : layer={CHOSEN_FWL['layer']}, pooling='{CHOSEN_FWL['pooling']}', FWL R²={CHOSEN_FWL['r2_residualized_calib']:+.4f}")
    print("  Next notebook should run both arms as an ablation.")

CHOSEN = CHOSEN_RAW if CHOSEN_RAW is not None else CHOSEN_FWL

# Persist
_selection_path = GEOMETRY_DIR / f"selection_gate_{CONFIG_HASH}.json"
_selection_path.write_text(json.dumps({
    "config_hash": CONFIG_HASH,
    "gate_branch": gate_branch,
    "chosen": CHOSEN,
    "chosen_raw": CHOSEN_RAW,
    "chosen_fwl": CHOSEN_FWL,
    "raw_selection_note": RAW_SELECTION_NOTE,
    "fwl_selection_note": FWL_SELECTION_NOTE,
    "n_raw_passing": len(raw_passing),
    "n_fwl_passing": len(fwl_passing),
    "n_failing": len(failing),
    "candidates": RESULTS,
}, indent=2, default=str))
print(f"\nSelection result → {_selection_path}")


## Honest reading

Depending on what the selection gate produced, the next step is one of four:

**Case A — no candidate passes the gate.** No usable prompt→target representation exists on base Qwen 0.5B for this data. That is a valid research finding: it says JEPA on templated NL-to-SQL at 0.5B is not going to work with base representations, regardless of hyperparameter tuning or auxiliary machinery. The next diagnostic move is running the same preflight on the SFT-trained checkpoint to see if fine-tuning has reshaped the target manifold enough to create pair-specific signal. If that also fails, the data itself is the bottleneck — the JEPA experiment needs a less templated dataset.

**IID/OOD readout.** If IID R² is decent but OOD R² is near zero, the mapping is family-shaped rather than abstract. That is different from "no signal" and should constrain the next notebook's claims.

**Case B — only the FWL-residualized gate passes.** The raw representation is dominated by the common direction + family-mean structure (the μ + α_g components), and the pair-specific residual s_i only becomes recoverable after those are partialled out. This is exactly the situation Residual JEPA is designed for. The next notebook builds raw and residualized arms, with residualized as the primary hypothesis.

**Case C — only the raw gate passes.** The representation naturally exposes pair-specific signal. Raw JEPA has a fair shot, while Residual JEPA is optional rather than required.

**Case D — both raw and residualized gates pass.** The next notebook should run both arms as a proper ablation to determine whether partialling-out helps or is unnecessary.

In all four cases, `04-outputs/geometry/selection_gate_{CONFIG_HASH}.json` holds the machine-readable decision via `gate_branch`: `none`, `raw_jepa`, `residual_jepa`, or `raw_and_residual`.
